# Probability Calibration

This notebook evaluates whether baseline model probabilities are reliable. It does not implement OOD detection, confidence estimation, evidence-quality scoring, or a Safety Gate.

## Why Calibration Matters

A baseline model can rank applicants reasonably while still producing unreliable probabilities. Future automated decision and safety-gating research depends on knowing whether predicted risk probabilities correspond to observed event rates.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.calibration import run_calibration_experiment

sns.set_theme(style='whitegrid')

## Baseline Probabilities

The raw baseline models output `P(financial_stress_event = 1)`. Calibration methods transform those probabilities using validation labels only.

## Calibration Methods and Protocol

Methods evaluated:

- Raw uncalibrated probabilities.
- Sigmoid / Platt calibration.
- Isotonic calibration.

Protocol: baseline models are fit on the training set; calibration mappings are fit on the validation set; the held-out test set is used only for final reporting.

In [ ]:
metrics = run_calibration_experiment(
    dataset_path=PROJECT_ROOT / 'data' / 'processed' / 'synthetic' / 'model_dataset.csv',
    output_dir=PROJECT_ROOT / 'experiments' / 'calibration',
    seed=42,
)
comparison = pd.read_csv(PROJECT_ROOT / 'experiments' / 'calibration' / 'calibration_comparison.csv')
predictions = pd.read_csv(PROJECT_ROOT / 'experiments' / 'calibration' / 'calibration_predictions.csv')
comparison

## Expected Calibration Error

ECE uses 10 fixed-width bins over `[0, 1]`. Each non-empty bin contributes its sample share multiplied by the absolute difference between mean predicted probability and observed event rate. Empty bins are ignored.

In [ ]:
comparison[['model', 'calibration_method', 'brier_score', 'log_loss', 'ece', 'roc_auc', 'pr_auc']]

## Reliability Diagrams

The diagonal line represents perfect calibration. Curves closer to the diagonal are better calibrated.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, model_name in zip(axes, ['logistic_regression', 'hist_gradient_boosting']):
    ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect calibration')
    for method, item in metrics['models'][model_name].items():
        curve = pd.DataFrame(item['calibration_diagnostics'])
        ax.plot(curve['mean_predicted_probability'], curve['observed_event_rate'], marker='o', label=method)
    ax.set_title(model_name)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Observed event rate')
    ax.legend()
fig.tight_layout()

## Probability Distributions Before and After Calibration

In [ ]:
grid = sns.displot(
    data=predictions,
    x='predicted_probability',
    hue='calibration_method',
    col='model',
    kind='hist',
    bins=40,
    element='step',
    stat='density',
    common_norm=False,
    height=4,
    aspect=1.2,
)
grid.set_axis_labels('Predicted probability', 'Density')

## Comparison and Interpretation

Calibration does not need to improve every metric. ROC-AUC and PR-AUC can remain unchanged for sigmoid calibration because ranking is preserved. Isotonic calibration can change ranking and may improve ECE while worsening log loss or ranking metrics.

In [ ]:
comparison.sort_values(['model', 'ece']).groupby('model').head(1)[['model', 'calibration_method', 'brier_score', 'log_loss', 'ece']]

## Limitations

The target is synthetic, so calibration quality here does not establish real-world credit-risk validity. Calibration is evaluated only on the generated dataset and should be revisited with external data or shifted populations before any safety-gating claims are made.